# AI Lab — Обучение с кастомными колбэками

Этот ноутбук демонстрирует:
1. Проверку GPU в контейнере
2. Загрузку модели через HuggingFace
3. Подключение кастомных колбэков (гистограммы весов/градиентов)
4. Запуск дообучения (QLoRA) с логированием в TensorBoard и Aim
5. Конвертацию логов TensorBoard → Aim

## 0. Установка зависимостей

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl aim tensorboard

## 1. Проверка GPU

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_mem / 1024**3
        print(f"  GPU {i}: {props.name} ({vram_gb:.1f} GB VRAM)")

## 2. Конфигурация обучения

Измените параметры ниже под вашу задачу:

In [ ]:
# ── Параметры ─────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"  # или meta-llama/Llama-3.1-8B
DATASET_PATH = "/home/jovyan/data"          # путь к датасету
OUTPUT_DIR = "/home/jovyan/output/run-01"
TB_LOG_DIR = "/home/jovyan/logs/tb"

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Обучение
NUM_EPOCHS = 3
BATCH_SIZE = 4
GRADIENT_ACCUMULATION = 4
LEARNING_RATE = 2e-4
LOGGING_STEPS = 10
SAVE_STEPS = 100

# Кастомные колбэки
HISTOGRAM_EVERY_N_STEPS = 50  # как часто логировать гистограммы

## 3. Загрузка модели (4-bit квантизация для QLoRA)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Модель загружена: {MODEL_NAME}")
print(f"Параметров: {model.num_parameters():,}")

## 4. Настройка LoRA

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Загрузка датасета

Пример с Alpaca-форматом. Замените на свой датасет.

In [ ]:
from datasets import load_dataset

# Пример: загрузка из HuggingFace Hub
# dataset = load_dataset("tatsu-lab/alpaca", split="train[:1000]")

# Пример: загрузка из локального JSON
# dataset = load_dataset("json", data_files="/home/jovyan/data/my_data.json", split="train")

# Демо-датасет для теста
dataset = load_dataset("tatsu-lab/alpaca", split="train[:500]")


def format_alpaca(example):
    if example.get("input"):
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        text = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return tokenizer(text, truncation=True, max_length=512, padding="max_length")


tokenized = dataset.map(format_alpaca, remove_columns=dataset.column_names)
print(f"Датасет: {len(tokenized)} примеров")

## 6. Подключение кастомных колбэков и запуск обучения

In [ ]:
import sys
sys.path.insert(0, "/home/jovyan/scripts")

from layer_histograms import LayerStatsCallback
from aim_logger import AimLoggerCallback
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    logging_dir=TB_LOG_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    report_to=["tensorboard"],
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    save_total_limit=3,
)

# Кастомные колбэки
layer_cb = LayerStatsCallback(
    log_every_n_steps=HISTOGRAM_EVERY_N_STEPS,
    log_grad_norm=True,
    filter_patterns=["lora"],  # логируем только LoRA-слои (экономия VRAM)
)

aim_cb = AimLoggerCallback(
    repo="/home/jovyan/logs/aim",  # или "aim://aim:53800" через сеть
    experiment="qlora-training",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized,
    callbacks=[layer_cb, aim_cb],
)

print("Запуск обучения...")
print(f"TensorBoard логи: {TB_LOG_DIR}")
print(f"Откройте http://localhost:6006 для мониторинга")

trainer.train()

## 7. Сохранение модели

In [ ]:
MERGED_DIR = "/home/jovyan/output/merged"

# Сохраняем LoRA-адаптер
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA-адаптер сохранён: {OUTPUT_DIR}")

# Сливаем LoRA с базовой моделью
merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Объединённая модель сохранена: {MERGED_DIR}")

## 8. Конвертация логов TensorBoard → Aim

Если обучение запускалось через LLaMA-Factory WebUI (без Aim-колбэка),
можно конвертировать уже записанные логи.

In [ ]:
from aim_logger import tb_to_aim

imported = tb_to_aim(
    tb_log_dir="/home/jovyan/logs/tb",
    repo="/home/jovyan/logs/aim",
    experiment="imported-from-llamafactory",
)
print(f"Импортировано {imported} метрик в Aim")
print("Откройте http://localhost:43800 для просмотра")

## 9. Быстрый тест модели

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=MERGED_DIR,
    tokenizer=MERGED_DIR,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

prompt = "### Instruction:\nОбъясни, что такое LoRA дообучение простыми словами.\n\n### Response:\n"

result = pipe(prompt, max_new_tokens=256, do_sample=True, temperature=0.7)
print(result[0]["generated_text"])